# Export Static Data

> Exports the `SolomonGeo` data to static JSON/GeoJSON files used by the Astro + MapLibre site in `web/`.

In [ ]:
#| default_exp export_static

The static site in `web/` has no Python server, so all of the data it needs is exported here once and
committed to `web/public/data/`. The export reuses the existing `SolomonGeo` class, so the numbers match the
Dash app exactly. Re-run this whenever the source data changes:

```bash
python -m SolomonIslandsDataMap.export_static            # build from the local testData/ files
python -m SolomonIslandsDataMap.export_static --aws      # build from the pickle stored in AWS S3
```

In [ ]:
#| export
import argparse
import json
import math
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
from fastcore.test import test_eq
from git import Repo

import SolomonIslandsDataMap.load_data as load_data
from SolomonIslandsDataMap.load_data import SolomonGeo

In [ ]:
#| export
def _repo_root() -> Path:
    "Absolute path of the root of the git repository"
    return Path(Repo('.', search_parent_directories=True).working_tree_dir)

## Loading the data

In [ ]:
#| export
def _test_geo(df:pd.DataFrame, # The dataframe to test
              geo:gpd.GeoDataFrame, # The geographies
             ):
    "Tests that every row of data has a matching geography (copied from `00_load_data`)"
    geo_df = geo.merge(df, left_on=['id', 'agg'], right_on=['core: id', 'core: agg'], how = 'outer', indicator = True)
    test_eq(sum(geo_df['_merge'] != 'both'), 0)

In [ ]:
#| export
def load_sol_geo(aws:bool = False, # Load the pickle from AWS S3 rather than building from `testData/`
                ) -> SolomonGeo:
    "Load the `SolomonGeo` object, either from AWS or from the local test data"
    if aws: return SolomonGeo.load_pickle(aws = True)
    # `read_test` relies on two names that only exist inside the 00_load_data notebook, provide them here
    load_data.fp = str(_repo_root() / 'testData') + '/'
    load_data.test_geo = _test_geo
    return SolomonGeo.read_test()

## Converting the data

In [ ]:
#| export
def _num(x, # A number (possibly numpy) to make json safe
         digits:int = 6, # Number of significant decimal places to keep for floats
        ):
    "Convert numpy numbers to plain python numbers, rounding floats and replacing NaN with None"
    if x is None: return None
    if isinstance(x, (np.integer, int)): return int(x)
    x = float(x)
    if math.isnan(x): return None
    if x.is_integer(): return int(x)
    return round(x, digits)

In [ ]:
#| export
def _round_coords(coords, digits:int = 5):
    "Recursively round geojson coordinates (5 decimals is roughly 1m)"
    if isinstance(coords[0], (int, float)): return [round(c, digits) for c in coords]
    return [_round_coords(c, digits) for c in coords]

In [ ]:
#| export
def export_geojson(sg:SolomonGeo, # The data object
                  ) -> dict: # Geo level -> GeoJSON feature collection
    "Simplified geographies for each geography level, the feature `name` is the location name"
    ret = {}
    for level in sg.geo_levels:
        geo = sg.geo.loc[sg.geo['agg'] == level].to_crs(4326)
        features = []
        for _, row in geo.iterrows():
            g = row.geometry.__geo_interface__
            features.append({"type": "Feature",
                             "properties": {"name": row['location'], "id": str(row['id'])},
                             "geometry": {"type": g['type'], "coordinates": _round_coords(json.loads(json.dumps(g['coordinates'])))}})
        ret[level] = {"type": "FeatureCollection", "features": features}
    return ret

In [ ]:
#| export
def export_census(sg:SolomonGeo) -> dict:
    """Census values for every location as `{geo: {location: {Total: [...], Proportion: [...]}}}`.
    The value arrays line up with `columns`, a list of `[variable, measure]` pairs."""
    columns = [[var, m] for var, measures in sg.census_vars.items() for m in measures]
    data = {}
    for level in sg.geo_levels:
        data[level] = {}
        for type_filter in sg.data_type:
            df = sg.get_census(geo_filter = level, var = None, type_filter = type_filter)
            df = df.loc[:, [tuple(c) for c in columns]]
            for loc, values in zip(df.index, df.values):
                data[level].setdefault(loc, {})[type_filter] = [_num(v) for v in values]
    return {"columns": columns, "data": data}

In [ ]:
#| export
def export_population(sg:SolomonGeo) -> dict:
    """Population projection rows (Province only). Each row is `[location, year, age, type, *values]`
    where the values line up with `measures`, the measures of the `Population` variable."""
    var = list(sg.population_vars.keys())[0]
    measures = sg.population_vars[var]
    out = {"variable": var, "measures": measures, "fields": ["location", "year", "age", "type"] + measures, "rows": []}
    pop = sg.population
    for type_filter in sg.data_type:
        df = pop.loc[pop['core']['type'] == type_filter]
        for loc, row in zip(df.index, df.itertuples(index = False)):
            r = pd.Series(row, index = df.columns)
            out['rows'].append([loc, int(r[('core', 'year')]), r[('Age', 'Age_Bracket')], type_filter]
                               + [_num(r[(var, m)], 8) for m in measures])
    return out

In [ ]:
#| export
def export_elections(sg:SolomonGeo) -> dict:
    "Election results, one row per candidate, plus the winning party per location"
    cols = ['Type', 'Year', 'Geo', 'loc_name', 'Candidate', 'candParty', 'Party', 'Total', 'Proportion', 'Order']
    rows = [{c: (_num(v) if c in ['Year', 'Total', 'Proportion', 'Order'] else v) for c, v in zip(cols, r)}
            for r in sg.elec.loc[:, cols].itertuples(index = False)]
    winners = [{"Type": r.Type, "Year": int(r.Year), "Geo": r.Geo, "loc_name": r.loc_name, "party": r._4}
               for r in sg.elec_wide.loc[:, ['Type', 'Year', 'Geo', 'loc_name', 'Winning Party']].itertuples(index = False)]
    return {"rows": rows, "winners": winners}

In [ ]:
#| export
def export_meta(sg:SolomonGeo) -> dict:
    "Lists used to build the filters on the site"
    pop = sg.population
    ages = (pop[('Age', 'Age_Bracket')].groupby(pop[('Age', 'Numerical_Bracket')]).first().tolist())
    return {"geoLevels": list(sg.geo_levels),
            "dataTypes": list(sg.data_type),
            "locations": {k: sorted(v.tolist()) for k, v in sg.locations.items()},
            "censusVars": sg.census_vars,
            "populationVars": sg.population_vars,
            "ages": ages,
            "popYears": [int(y) for y in sg.pop_years],
            "elections": sg.elections,
            "elecYears": {k: [int(y) for y in v] for k, v in sg.elec_years.items()},
            "partyColors": sg.colormap}

## Writing the files

In [ ]:
#| export
def export_static(sg:SolomonGeo, # The data object
                  out_dir:Path = None, # Folder to write to, defaults to `web/public/data`
                 ) -> list: # The paths written
    "Write all of the static data files used by the website"
    out_dir = Path(out_dir) if out_dir is not None else _repo_root() / 'web' / 'public' / 'data'
    out_dir.mkdir(parents = True, exist_ok = True)
    files = {"meta.json": export_meta(sg), "census.json": export_census(sg),
             "population.json": export_population(sg), "elections.json": export_elections(sg)}
    for level, fc in export_geojson(sg).items(): files[f"geo_{level.lower()}.geojson"] = fc
    written = []
    for name, obj in files.items():
        p = out_dir / name
        p.write_text(json.dumps(obj, separators = (',', ':'), ensure_ascii = False))
        written.append(p)
    return written

In [ ]:
#| export
def main():
    parser = argparse.ArgumentParser(description = "Export the Solomon Islands data to static files for the website")
    parser.add_argument('--aws', action = 'store_true', help = "Load the data pickle from AWS instead of testData/")
    parser.add_argument('--out', default = None, help = "Output folder, defaults to web/public/data")
    args = parser.parse_args()
    for p in export_static(load_sol_geo(aws = args.aws), args.out): print(f"{p.stat().st_size/1e3:>9.1f} kB  {p}")

In [ ]:
#| export
if __name__ == "__main__" and not hasattr(__builtins__, '__IPYTHON__'): main()

Run the export from the notebook (not run by `nbdev_test`, as it writes files):

In [ ]:
#| eval: false
for p in export_static(load_sol_geo()): print(p)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()